In [43]:
import polars as pl
import catboost as cb
import os

path_train = os.path.join(os.getcwd(), "..", "data","train_main_features.parquet")
path_test = os.path.join(os.getcwd(),"..","data","test_main_features.parquet")
path_target = os.path.join(os.getcwd(),"..","data","train_target.parquet")
train = pl.read_parquet(path_train)
test = pl.read_parquet(path_test)
target = pl.read_parquet(path_target)

print(train.head(5))

target_columns = [c for c in target.columns if c != "customer_id"]


shape: (5, 200)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ customer_ ┆ cat_featu ┆ cat_featu ┆ cat_featu ┆ … ┆ num_featu ┆ num_featu ┆ num_featu ┆ num_feat │
│ id        ┆ re_1      ┆ re_2      ┆ re_3      ┆   ┆ re_129    ┆ re_130    ┆ re_131    ┆ ure_132  │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ i32       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1000001   ┆ 1.0       ┆ 0.0       ┆ 2.0       ┆ … ┆ -0.107666 ┆ -0.418616 ┆ null      ┆ null     │
│ 1000002   ┆ 1.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ -0.170724 ┆ -0.805771 ┆ -0.397803 ┆ -0.37373 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 4        │
│ 1000003   ┆ 1.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ -0.170724 ┆ -0.602005

In [ ]:
target_summ = []
for c in target_columns:
    vals = target.select(pl.col(c).drop_nulls().unique().sort()).to_series().to_list()
    target_summ.append({
        "target": c,
        "n_unique": len(vals),
        "unique_values_preview": vals[:10]
    })

print(pl.DataFrame(target_summ))

shape: (41, 3)
┌─────────────┬──────────┬───────────────────────┐
│ target      ┆ n_unique ┆ unique_values_preview │
│ ---         ┆ ---      ┆ ---                   │
│ str         ┆ i64      ┆ list[f64]             │
╞═════════════╪══════════╪═══════════════════════╡
│ target_1_1  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_2  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_3  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_4  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_5  ┆ 2        ┆ [0.0, 1.0]            │
│ …           ┆ …        ┆ …                     │
│ target_9_5  ┆ 2        ┆ [0.0, 1.0]            │
│ target_9_6  ┆ 2        ┆ [0.0, 1.0]            │
│ target_9_7  ┆ 2        ┆ [0.0, 1.0]            │
│ target_9_8  ┆ 2        ┆ [0.0, 1.0]            │
│ target_10_1 ┆ 2        ┆ [0.0, 1.0]            │
└─────────────┴──────────┴───────────────────────┘


In [36]:
target_balance = []

for c in target_columns:
    vc = (
        target.group_by(c)
        .len()
        .with_columns((pl.col("len") / pl.col("len").sum()).alias("share"))
        .sort(c)
    )
    
    zeros = vc.filter(pl.col(c) == 0).select("share").to_series().to_list()
    ones = vc.filter(pl.col(c) == 1).select("share").to_series().to_list()
    
    target_balance.append({
        "target": c,
        "share_0": zeros[0] if len(zeros) else None,
        "share_1": ones[0] if len(ones) else None
    })

balance_df = pl.DataFrame(target_balance).sort("share_1")
print(balance_df)

shape: (41, 3)
┌─────────────┬──────────┬──────────┐
│ target      ┆ share_0  ┆ share_1  │
│ ---         ┆ ---      ┆ ---      │
│ str         ┆ f64      ┆ f64      │
╞═════════════╪══════════╪══════════╡
│ target_2_8  ┆ 0.999889 ┆ 0.000111 │
│ target_2_7  ┆ 0.999697 ┆ 0.000303 │
│ target_6_5  ┆ 0.999441 ┆ 0.000559 │
│ target_3_3  ┆ 0.998813 ┆ 0.001187 │
│ target_2_3  ┆ 0.998612 ┆ 0.001388 │
│ …           ┆ …        ┆ …        │
│ target_3_2  ┆ 0.902591 ┆ 0.097409 │
│ target_3_1  ┆ 0.901627 ┆ 0.098373 │
│ target_8_1  ┆ 0.897504 ┆ 0.102496 │
│ target_9_6  ┆ 0.776928 ┆ 0.223072 │
│ target_10_1 ┆ 0.684948 ┆ 0.315052 │
└─────────────┴──────────┴──────────┘


In [41]:
train_full = train.join(target, on="customer_id", how="inner")
print(train_full.head(5))

shape: (5, 241)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ customer_ ┆ cat_featu ┆ cat_featu ┆ cat_featu ┆ … ┆ target_9_ ┆ target_9_ ┆ target_9_ ┆ target_1 │
│ id        ┆ re_1      ┆ re_2      ┆ re_3      ┆   ┆ 6         ┆ 7         ┆ 8         ┆ 0_1      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ i32       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1000001   ┆ 1.0       ┆ 0.0       ┆ 2.0       ┆ … ┆ 1.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ 1000002   ┆ 1.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 0.0      │
│ 1000003   ┆ 1.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ 1.0      │
│ 1000004   ┆ 1.0       ┆ 0.0       ┆ 2.0       ┆ … ┆ 1.0       ┆ 0.0      

In [ ]:
from sklearn.model_selection import train_test_split
X_train, y_train = train_test_split(train_full)